# Text Recognition — Fine-tune TrOCR

Распознавание текста на вырезанных регионах страниц учебников.

**Pipeline:**
1. Загрузка датасета (наши данные + HuggingFace кириллические)
2. Визуализация примеров
3. Fine-tune `microsoft/trocr-base-printed`
4. Мониторинг CER/WER по эпохам
5. Визуализация предсказаний vs ground truth
6. Анализ ошибок

**Железо:** RTX 3080ti — batch=8, fp16=True

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../..'))

from pathlib import Path
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import numpy as np

DATA_DIR = Path('../../data/recognition')
OUTPUT_DIR = Path('runs/recognition')
BASE_MODEL = 'microsoft/trocr-base-printed'
EPOCHS = 10
BATCH = 8
FP16 = True   # False если нет GPU с поддержкой fp16
DEVICE = 'cuda'  # 'mps' для Mac M4, 'cpu' для CPU-only

print(f'DATA_DIR exists: {DATA_DIR.exists()}')
meta = DATA_DIR / 'metadata.jsonl'
if meta.exists():
    with open(meta) as f:
        n = sum(1 for l in f if l.strip())
    print(f'Примеров в датасете: {n}')
else:
    print('metadata.jsonl не найден — запустите ocr/data/prepare.py')

## 1. Подготовка данных

Запустить если данных нет:

In [ ]:
if not (DATA_DIR / 'metadata.jsonl').exists():
    # Наши данные из учебников
    !python -m ocr.data.prepare \
        --books_dir ../../books \
        --output_dir ../../data/recognition \
        --mode recognition \
        --pages_per_book 30

    # Дополнительно: кириллические датасеты с HuggingFace
    !python -m ocr.data.cyrillic \
        --dataset ai-forever/ocr \
        --output_dir ../../data/recognition \
        --max_samples 5000
else:
    print('Данные уже подготовлены')

## 2. Визуализация примеров датасета

In [ ]:
samples = []
meta_path = DATA_DIR / 'metadata.jsonl'
if meta_path.exists():
    with open(meta_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))

n_show = min(6, len(samples))
if n_show > 0:
    fig, axes = plt.subplots(n_show, 1, figsize=(16, 3 * n_show))
    if n_show == 1:
        axes = [axes]
    for ax, s in zip(axes, samples[:n_show]):
        img = Image.open(s['image'])
        ax.imshow(img)
        ax.set_title(f"GT: {s['text'][:120]}", fontsize=9, loc='left')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Нет примеров')

## 3. Обучение

In [ ]:
from ocr.recognition.train import train

best_model_dir = train(
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    model_name=BASE_MODEL,
    epochs=EPOCHS,
    batch_size=BATCH,
    learning_rate=5e-5,
    max_target_length=128,
    val_ratio=0.1,
    fp16=FP16,
    seed=42,
)

print(f'Модель сохранена: {best_model_dir}')

## 4. CER / WER по эпохам

In [ ]:
import json

# Читаем trainer_state.json для истории метрик
state_files = list(OUTPUT_DIR.rglob('trainer_state.json'))
if state_files:
    with open(state_files[0]) as f:
        state = json.load(f)
    
    log_history = state.get('log_history', [])
    eval_logs = [l for l in log_history if 'eval_cer' in l]
    train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
    
    if eval_logs:
        epochs_e = [l['epoch'] for l in eval_logs]
        cer_vals = [l['eval_cer'] for l in eval_logs]
        wer_vals = [l.get('eval_wer', None) for l in eval_logs]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        axes[0].plot(epochs_e, cer_vals, 'b-o', label='CER')
        axes[0].set_title('Character Error Rate (↓ лучше)')
        axes[0].set_xlabel('Epoch')
        axes[0].grid(True, alpha=0.3)
        
        if any(w is not None for w in wer_vals):
            axes[1].plot(epochs_e, wer_vals, 'r-o', label='WER')
            axes[1].set_title('Word Error Rate (↓ лучше)')
            axes[1].set_xlabel('Epoch')
            axes[1].grid(True, alpha=0.3)
        
        if train_logs:
            steps = [l['step'] for l in train_logs]
            losses = [l['loss'] for l in train_logs]
            axes[2].plot(steps, losses, 'g-', alpha=0.7, label='Train Loss')
            axes[2].set_title('Train Loss')
            axes[2].set_xlabel('Step')
            axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f'Лучший CER: {min(cer_vals):.4f}')
        if any(w is not None for w in wer_vals):
            print(f'Лучший WER: {min(w for w in wer_vals if w is not None):.4f}')
else:
    print('trainer_state.json не найден')

## 5. Предсказания vs Ground Truth

In [ ]:
from ocr.recognition.model import TextRecognizer

best_dir = OUTPUT_DIR / 'best'
model_path = str(best_dir) if best_dir.exists() else BASE_MODEL
recognizer = TextRecognizer(model_path, device=DEVICE)

# Берём val-примеры
val_meta = DATA_DIR / 'metadata_val.jsonl'
if not val_meta.exists():
    val_meta = DATA_DIR / 'metadata.jsonl'

val_samples = []
with open(val_meta, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            val_samples.append(json.loads(line))

import random
random.shuffle(val_samples)
show_samples = val_samples[:8]

fig, axes = plt.subplots(len(show_samples), 1, figsize=(16, 3 * len(show_samples)))
if len(show_samples) == 1:
    axes = [axes]

for ax, s in zip(axes, show_samples):
    img = Image.open(s['image'])
    pred = recognizer.predict(img)
    gt = s['text']
    
    # Подсвечиваем совпадения/несовпадения
    match = pred.strip() == gt.strip()
    color = 'green' if match else 'red'
    
    ax.imshow(img)
    ax.set_title(
        f"GT:   {gt[:100]}\nPred: {pred[:100]}",
        fontsize=8, loc='left', color=color
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Анализ ошибок — топ-10 худших примеров

In [ ]:
import evaluate

cer_metric = evaluate.load('cer')

errors = []
for s in val_samples[:100]:  # оцениваем первые 100 для скорости
    img = Image.open(s['image'])
    pred = recognizer.predict(img)
    gt = s['text']
    cer = cer_metric.compute(predictions=[pred], references=[gt])
    errors.append({'gt': gt, 'pred': pred, 'cer': cer, 'image': s['image']})

errors.sort(key=lambda x: x['cer'], reverse=True)

print('Топ-10 худших примеров:')
print(f'{"CER":>8}  {"GT":50}  {"Pred"}')
print('-' * 120)
for e in errors[:10]:
    print(f"{e['cer']:8.4f}  {e['gt'][:50]:50}  {e['pred'][:50]}")

mean_cer = sum(e['cer'] for e in errors) / len(errors)
print(f'\nСредний CER на {len(errors)} примерах: {mean_cer:.4f}')